#### Create automated_metrics.db. Will be around 5GB upon notebook running completion. Recommended to use the compressed (zipped) DB.

In [2]:
import sqlite3
import os
import re
import json

Set up DB and tables

In [3]:
con = sqlite3.connect("automated_metrics.db")
con.execute("PRAGMA foreign_keys = 1")
cur = con.cursor()

cur.execute("CREATE TABLE context(context_id TEXT PRIMARY KEY, " \
"context TEXT, " \
"justice TEXT, " \
"turn_text TEXT, " \
"transcript_id TEXT, " \
"FOREIGN KEY (transcript_id) REFERENCES transcript(transcript_id) ON DELETE CASCADE);")

cur.execute("CREATE TABLE transcript(transcript_id TEXT PRIMARY KEY, " \
"case_facts TEXT, " \
"legal_question TEXT, " \
"conclusion TEXT);")

Populate transcripts table

In [4]:
folder_path = "./case_briefs/"

def get_case_docket_ids():
    docket_ids = []
    for file in os.listdir(folder_path):
        match = re.search(r"^\d{4}\.[a-zA-Z0-9_-]+(?=\.json)", file)
        if match:
            docket_ids.append(match.group())

    return docket_ids

def strip_paragraph_brackets(text):
    return re.sub("</?p[^>]*>", "", text)

for case in get_case_docket_ids():
    with open(os.path.join(folder_path, case + ".json")) as file:
        case_data = json.load(file)
        facts = "" if not case_data["facts_of_the_case"] else strip_paragraph_brackets(case_data["facts_of_the_case"])
        question = "" if not case_data["question"] else strip_paragraph_brackets(case_data["question"])
        conclusion = "" if not case_data["conclusion"] else strip_paragraph_brackets(case_data["conclusion"])

        cur.execute("INSERT INTO transcript(transcript_id, case_facts, legal_question, conclusion) VALUES(?, ?, ?, ?)", (case, facts, question, conclusion))
        con.commit()

Populate context table

In [7]:
folder_path = "./cleaned_transcripts/restructured_overlaps_removed/"
def get_transcript_dockets():
    dockets = []
    for file in os.listdir(folder_path):
        match = re.search(r"^\d{4}\.[a-zA-Z0-9_-]+(?=-transcript.json)", file)
        if match is None:
            print(file)
        dockets.append(match.group())
    return dockets

def ensure_speaker_side_and_role_integrity(turn):
    # assigns an advocate's side to "advocate" if it doesn't exist
    if turn["speaker"]["side"] == None:
        turn["speaker"]["side"] = turn["speaker"]["role"]

def is_previous_speaker_advocate(section, current_turn_idx):
    if current_turn_idx == 0:
        return False
    previous_turn = section["turns"][current_turn_idx - 1]
    return previous_turn["speaker"]["role"] == "advocate"

def process_transcript_into_context(docket, transcript_data):
    sections = transcript_data["sections"]
    for section_idx, section in enumerate(sections):
        current_context = []
        for turn_idx, turn in enumerate(section["turns"]):
            if turn["speaker"]["role"] == "advocate":
                ensure_speaker_side_and_role_integrity(turn)
            
            if turn["speaker"]["role"] == "scotus_justice" and is_previous_speaker_advocate(section, turn_idx):
                # create a new context entry when it's a justice's turn to speak
                context_id = f"{docket}_s{section_idx}_t{turn_idx}"
                transcript_id = docket
                justice = turn["speaker"]["name"]
                turn_text = turn["text"]
                context = json.dumps(current_context)
                cur.execute("INSERT INTO context(context_id, context, justice, turn_text, transcript_id) VALUES(?, ?, ?, ?, ?)", (context_id, context, justice, turn_text, transcript_id))
                con.commit()
            
            current_context.append(turn)

for docket in get_transcript_dockets():
    with open(os.path.join(folder_path, docket + "-transcript.json")) as file:
        transcript_data = json.load(file)
    process_transcript_into_context(docket, transcript_data)

Compress database ~5GB and 235833 lines of context entries uncompressed

In [8]:
import zipfile

with zipfile.ZipFile('compressed_metrics_db.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('automated_metrics.db', arcname='automated_metrics.db')